### Create Test/Train Sets

In [ ]:
# import json
# import pandas as pd
# def import_jsonl(filepath):
#     total_data = []
#     with open(filepath) as datafile:
#         for line in datafile:
#             item = json.loads(line)
#             total_data.append({
#                 'text': item['text']
#             })

#     print(len(total_data))
#     return total_data

# data = import_jsonl("../corpus/data/twitter_training_data.jsonl")


# df = pd.DataFrame.from_dict(data)
# df = df.drop_duplicates()
# df["ID"] = pd.util.hash_pandas_object(df["text"])
# df.to_csv("../corpus/data/twitter_train_data.csv", index=False)

#### Pull Labeled Data from Prodigy

In [ ]:
# from prodigy.components.db import connect
# from collections import defaultdict
# from pprint import pprint
# import pandas as pd
# import random 


# db = connect()
# examples = db.get_dataset("twitter") #dedupe_ids

# test_data = []
# for eg in examples:
#     if eg["answer"] != "ignore":  # you probably want to exclude ignored/rejected answers?
#        test_data.append(
#            {
#                'ID': eg['_task_hash'],
#                'text': eg['text'],
#                'label': eg['answer']
#                })
# random.shuffle(test_data)
# pprint(test_data)

### Transform test dictionary into Dataframe
# test_df = pd.DataFrame.from_dict(test_data)
# print(len(test_df))

# test_df.to_csv("../corpus/data/twitter_test_data.csv", index=False)

#### Pull unlabeled data from the Twitter Scraped Data

In [ ]:
import pandas as pd
### Read in the train data for Label Functions
# dfx = pd.read_csv("../corpus/data/reddit/crypto_data_updated.csv", error_bad_lines=False, engine='python')
#df_test = pd.read_csv("../corpus/data/twitter_test_data.csv")
# ###
# ### Change the labels for df_test
df_test["label"] = df_test["label"].apply(lambda x: 1 if x == "reject" else 0)
Y_test = df_test.label.values
### 
df3 = dfx.merge(df_test, on='text')
df_train = dfx[~dfx['text'].isin(df3['text'])]
### Use df_train as the basis for the label_functions
df_train = df_train.sample(frac=1, random_state=123).reset_index(drop=True)
df_train[["text", "_id"]].sample(20, random_state=2)

In [ ]:
# from sklearn import model_selection


# df_train, df_valid = model_selection.train_test_split(train, 
#     test_size=0.10, shuffle=True, random_state=43, stratify=test.label.values
# )

In [ ]:
# For clarity, we define constants to represent the class labels for spam, ham, and abstaining.
IGNORE = -1
RELEVANT = 0
IRRELEVANT = 1

In [ ]:
from snorkel.labeling import labeling_function


@labeling_function()
def presale(x):
    return IRRELEVANT if "presale" in x.text.lower() else IGNORE


@labeling_function()
def giveaway(x):
    return IRRELEVANT if "giveaway" in x.text.lower() else IGNORE

In [ ]:
from snorkel.labeling import PandasLFApplier

lfs = [presale, giveaway]

applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=df_train)

In [ ]:
coverage_presale, coverage_giveaway = (L_train != IGNORE).mean(axis=0)
print(f"presale coverage: {coverage_presale * 100:.1f}%")
print(f"giveaway coverage: {coverage_giveaway * 100:.1f}%")

In [ ]:
from snorkel.labeling import LFAnalysis

LFAnalysis(L=L_train, lfs=lfs).lf_summary()

In [ ]:
df_train.iloc[L_train[:, 1] == IRRELEVANT].sample(10, random_state=1)

In [ ]:
from snorkel.analysis import get_label_buckets

buckets = get_label_buckets(L_train[:, 0], L_train[:, 1])
df_train.iloc[buckets[(IGNORE, IRRELEVANT)]].sample(10, random_state=1)

In [ ]:
from transformers import AutoModel
from transformers import pipeline
from snorkel.preprocess import preprocessor


model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
model = AutoModel.from_pretrained(model_name)

sentiment_task = pipeline("sentiment-analysis", model=model_name, tokenizer=model_name)


@preprocessor(memoize=True)
def sentiment_selector(x):
    output = sentiment_task(x.text)
    x.sentiment = 'NEG' if 'Negative' in output[0]['label'] else 'POS'
    return x


In [ ]:
@labeling_function(pre=[sentiment_selector])
def huggingface_sentiment(x):
    return RELEVANT if x.sentiment == 'POS' else IGNORE

In [ ]:
lfs = [huggingface_sentiment]

applier = PandasLFApplier(lfs)
L_train = applier.apply(df_train)

In [ ]:
LFAnalysis(L_train, lfs).lf_summary()

In [ ]:
import re
@labeling_function()
def regex_ether_contract(x):
    return RELEVANT if re.search(r"^0x[a-fA-F0-9]{20}$", x.text, flags=re.I) else IRRELEVANT

In [ ]:
from snorkel.labeling import LabelingFunction


def keyword_lookup(x, keywords, label):
    if any(word in x.text.lower() for word in keywords):
        return label
    return IGNORE


def make_keyword_lf(keywords, label=IRRELEVANT):
    return LabelingFunction(
        name=f"keyword_{keywords[0]}",
        f=keyword_lookup,
        resources=dict(keywords=keywords, label=label),
    )


"""Irrelevant comments talk about 'awesome', 'excited', etc."""
keyword_excited = make_keyword_lf(keywords=["awesome", "excited", "moon"])

keyword_presale = make_keyword_lf(keywords=["presale"])


### LAbels for Hashtags
crypto_keywords = [
    'cryptoscam', 
    'dump', 
    'dumping', 
    'fake', 
    'fishy', 
    'fraud', 
    'front running', 
    'givaway scams', 
    'hijack', 
    'honey pot', 
    'illicit', 
    'legit?', 
    'lost', 
    'phishing', 
    'pump and dump', 
    'retirement', 
    'rug pull', 
    'rug-pull', 
    'rugged', 
    'scam',
    'scamalert', 
    'scammed', 
    'scammer', 
    'shitcoin', 
    'stolen', 
    'stupidity', 
    'trusted', 
    'unreal', 
    'warning',
    '#phishing']

keyword_crypto = make_keyword_lf(keywords=crypto_keywords, label=RELEVANT)

In [ ]:
lfs = [
    keyword_excited,
    keyword_presale,
    keyword_crypto,
   # regex_ether_contract,
    huggingface_sentiment,
]

In [ ]:
df_train

In [ ]:
applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=df_train)
L_test = applier.apply(df=df_test)

In [ ]:
LFAnalysis(L=L_train, lfs=lfs).lf_summary()

In [ ]:
from snorkel.labeling.model import LabelModel

label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(L_train=L_train, n_epochs=500, log_freq=100, seed=123)

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline


def plot_label_frequency(L):
    plt.hist((L != IGNORE).sum(axis=1), density=True, bins=range(L.shape[1]))
    plt.xlabel("Number of labels")
    plt.ylabel("Fraction of dataset")
    plt.show()


plot_label_frequency(L_train)

In [ ]:
from snorkel.labeling.model import MajorityLabelVoter

majority_model = MajorityLabelVoter()
preds_train = majority_model.predict(L=L_train)

majority_acc = majority_model.score(L=L_test, Y=Y_test, tie_break_policy="random")[
    "accuracy"
]
print(f"{'Majority Vote Accuracy:':<25} {majority_acc * 100:.1f}%")

label_model_acc = label_model.score(L=L_test, Y=Y_test, tie_break_policy="random")[
    "accuracy"
]
print(f"{'Label Model Accuracy:':<25} {label_model_acc * 100:.1f}%")

In [ ]:
def plot_probabilities_histogram(Y):
    plt.hist(Y, bins=10)
    plt.xlabel("Probability of Relevant Tweet")
    plt.ylabel("Number of data points")
    plt.show()


probs_train = label_model.predict_proba(L=L_train)
plot_probabilities_histogram(probs_train[:, RELEVANT])

In [ ]:
from snorkel.labeling import filter_unlabeled_dataframe

df_train_filtered, probs_train_filtered = filter_unlabeled_dataframe(
    X=df_train, y=probs_train, L=L_train
)

In [ ]:
train = "../corpus/data/twitter_training_data.csv"
# test = "../corpus/data/twitter_test_data.csv"

# import pandas as pd

df_train = pd.read_csv(train)
df_train['label'] = "UNKNOWN"
df_train = df_train.sample(frac=0.20).reset_index(drop=True)


# df_test = pd.read_csv(test)
# df_test.label = df_test.label.apply(lambda x: "RELEVANT" if x == "accept" else "IRRELEVANT")
# df_test = df_test.sample(frac=1).reset_index(drop=True)

# # Ground truth Data
# ground_truth = df_test.iloc[0:50]
# # Data to Append
# test_append = df_test.iloc[51:100]


# # Final test data to export
# test_final = df_test.iloc[101:]



In [ ]:
#df_test.label = df_test.label.apply(lambda x: "RELEVANT" if x == "accept" else "IRRELEVANT")
ground_truth.head()

In [ ]:
#train_to_split = df_train.merge(test_append)
# train_to_split = pd.concat([test_append,df_train])
#train_to_split = train_to_split.sample(frac=1).reset_index(drop=True)
#train_to_split.head()

In [ ]:
from sklearn import model_selection

train_final, valid_final = model_selection.train_test_split(
    train_to_split, test_size=0.15, random_state=43, stratify=train_to_split.label.values
)

train_final = train_final.reset_index(drop=True)
valid_final = valid_final.reset_index(drop=True)

In [ ]:
len(train_final)

In [ ]:
# Train 75%
train_final.to_csv("../corpus/data/toy/toy_train.csv", index=False, encoding="utf-8")
# Valid 15%
valid_final.to_csv("../corpus/data/toy/toy_valid.csv", index=False, encoding="utf-8") 
# Test 15%
test_final.to_csv("../corpus/data/toy/toy_test.csv", index=False, encoding="utf-8")  
# Ground Truth
ground_truth.to_csv("../corpus/data/toy/toy_groundtruth.csv", index=False, encoding="utf-8")

In [ ]:
class_names = ["employment", "services", "stock", "loan"]

for class_name in class_names:
    
    @labeling_function(name=f"NB-{class_name}-in-text", resources=dict(class_name=class_name))
    def lf(x, class_name):

        if class_name.upper() in x.text:
            return class_name
        else: 
            return "UNKNOWN"

    dataset.save(lf, label_str=class_name)

In [ ]:
def first_dollar_amount(text):
    import re
    regex = re.compile(r"\$([0-9]{1,3},([0-9]{3},)*[0-9]{3}|[0-9]+)")
    monetary_amount_string = ["".join(amt) for amt in regex.findall(text)]
    monetary_amounts = [float(re.sub(r"[^\d\.]", "", amt)) for amt in monetary_amount_string]
    
    if monetary_amounts == []:
        return 0
    else:
        return monetary_amounts[0]


df["first_dollar_amount"] = df['text'].apply(first_dollar_amount)
dataset.plot_distributions(
    df, 
    "first_dollar_amount", 
    label_field="label",
    hist_kwargs=dict(range=(0,1e6)),
)

In [ ]:
@labeling_function(name="NB-ServicesAgreement")
def lf(x):
    if "Services Agreement" in x.text:
        return "services"
    else: 
        return "UNKNOWN"

dataset.save(lf, label_str="services")

In [ ]:
crypto = pd.read_csv("https://raw.githubusercontent.com/matthewkmoore/CryptoCurrency-Media-Analysis/master/coin_market_caps.csv")

In [ ]:
len(crypto)

In [ ]:
lists = ['cryptoscam', 
    'dump', 
    'dumping', 
    'fake', 
    'fishy', 
    'fraud', 
    'front running', 
    'givaway scams', 
    'hijack', 
    'honey pot', 
    'illicit', 
    'legit?', 
    'lost', 
    'phishing', 
    'pump and dump', 
    'retirement', 
    'rug pull']

# for l in lists:
#     {'label': }
# authors = {key: ' '.join(key.split()) for key in keys}

In [ ]:
# {"label": "Scam", "pattern":
data = '[{"lower": '.join(lists[7].split(' '))

In [ ]:
test = []
tester = []
for l in lists.split(' '):
    tester.append({'lower':l})
test.append({"label":"CYBER","pattern": tester})
print(test)

In [ ]:
def pattern_maker(text, label_name):
    tester = []
    test = []
    for l in text.split(' '):
        tester.append({'lower':l})
    test.append({"label":label_name,"pattern": tester})

    return test


# patterns = []

# for l in lists:
#     patterns.append(pattern_maker(l, "CYBER"))
# print(patterns)


In [ ]:
#crypto = crypto.drop_duplicates(subset='symbol')

cryptos = list(crypto['symbol'])

In [ ]:

patterns = []

for cryp in cryptos:
    patterns.append(pattern_maker(cryp, "CRYPTO")[0])
print(patterns[0:10])

In [ ]:
patterns[0]

In [ ]:
import srsly 
srsly.write_jsonl("../patterns/test_file.jsonl", patterns)

In [ ]:
for p in patterns:
    srsly.write_jsonl("../patterns/test_file.jsonl", p)
    # print(p[0])

In [ ]:
import typer
import random
from prodigy.components.db import connect
#import srsly
from pathlib import Path
from spacy.util import get_words_and_spaces
from spacy.tokens import Doc, DocBin
import spacy


def preprocess(annotated):
    """use this to get a binanry classification dataset
    Args:
        annotated (_str_): _description_
        dataset (_list_): imported list of annotated data from prodigy
    Returns:
        _type_: list of dictionary
    """
    db = connect()
    dataset = db.get_dataset(annotated) # name of the prodigy labeled data
    processed = []
    for data in dataset:
        if data['answer'] != 'ignore':
            processed.append({
                'text': data['text'],
                'labels': 'RELEVANT' if 'accept' in data['answer'] else 'IRRELEVANT'
            })
    return processed


def make_spacy(nlp, output, records, categories):
    #nlp = spacy.blank("en")
    doc_bin = DocBin()
    data_tuples = ((eg["text"], eg) for eg in records)
    for doc, eg in nlp.pipe(data_tuples, as_tuples=True):
        doc.cats = {category:0 for category in categories}
        doc.cats[eg["labels"]] = 1
        doc_bin.add(doc)
    return doc_bin.to_disk(output)

In [ ]:
def preprocess(annotated):
    """use this to get a binanry classification dataset
    Args:
        annotated (_str_): _description_
        dataset (_list_): imported list of annotated data from prodigy
    Returns:
        _type_: list of dictionary
    """
    db = connect()
    dataset = db.get_dataset(annotated) # name of the prodigy labeled data
    processed = []
    for data in dataset:
        if data['answer'] != 'ignore':
            processed.append({
                'text': data['text'],
                'labels': 'RELEVANT' if 'accept' in data['answer'] else 'IRRELEVANT'
            })
    return processed

In [ ]:
twitter2 = preprocess("merged_twitter")

In [ ]:
len(twitter2)

In [ ]:

db = connect()
datasets = ['merged_twitter']
#dataset = db.get_dataset(annotated) # name of the prodigy labeled data
#datasets = annotated
examples = []
processed = []
for data in datasets:
    examples += db.get_dataset(data)
    for example in examples:

        if example['answer'] != 'ignore':
            processed.append({
                'text': example['text'],
                'labels': 'RELEVANT' if 'accept' in example['answer'] else 'IRRELEVANT'
            })
#return processed

In [ ]:
datasets = ['twitter', 'twitter2']
#dataset = db.get_dataset(annotated) # name of the prodigy labeled data
#datasets = annotated
examples = []
processed = []
for data in datasets:
    examples += db.get_dataset(data)

In [ ]:
phished

#phishing
#bitcoinscam

Crypto Scam

Bitcoin scam

'rug pull' scam


'0xc9f109549685ab5ea7a91031d677bcdd2d1611fe'


# $1.1 million 'rug pull' scam   money ----- scam term

In [ ]:
import spacy
nlp = spacy.load("../training/model-best")

In [ ]:
texts = ["""@SuperlativeApes
 you blocked me from discord for speaking truth. I put over 25k into your project only to see you run with all the money you took from mints and royalties. Fuck you. 🙂 that is all. Burn in hell with the millions you stole from everyone 🌈💩#rugpull""",

        "This movie was terrifying, I jumped out of my seat I was so scared, I never want to watch this again", 
        """We've Flagged the TheTravelers which was using @tzehoo1
 art for a fake mint and potential Phishing Scam.
That same Account is now 🚨@LokiiNFT
 - Which is by default a #RUGPULL @NFT_Awareness
 @NFT_Annalea
 @VoodooNFTs
 @Ammo9168
 @ErrorZer0invest
 @NFT_watchdog
  #NFTs #NFTcommunity""", """⚠️ Crypto Scam List ⚠️

All same red flags 🚩
Appears to be the same #scam team as Terk if you follow their address:

🚫 Terk
🚫 MetaUFO
🚫 MetaWar
🚫 Metarobox
🚫 Bull100x
🚫 BabyDogeDooDoo
🚫 Rottweiler
🚫 Web3Coin
🚫 Rise Moon
🚫 Soccer Fan

Plz RT!

#ScamAlert
#CommunityAlert""", 
"""🚨 SCAM /RIP OFF ALERT 🚨 
<< RT to warn others>> @Marcoswydd
 posts GAWs & codes, so I trusted. Paid $200 for a Wildcat code and did not deliver as promised. At the time of this tweet, he is still trying to sell codes.
See screenshots ⬇️
#ripoff  #SCAM #Fortnite""", 
"""Pirvulescu Vlad
@vlad_pirvulescu
·
May 4
🔥 1 week #Scam Minotaurs  Giveaway 🔥

‼️2 prizes‼️
🥇1 EGLD
🥈1.000.000  $lkmex

✅Rules✅
1️⃣ Follow @Vlad_pirvulescu

2️⃣ Like + RT
3️⃣ Tag 7 friends
#Giveaway
They ban people on Discord for telling the truth! 
They buy the Minotaurs to make volume and to increase the floor price""", 
"""⚠️ Scam warning ⚠️ !
@zapper_fi
 are NOT holding
a token presale. They haven't even decided if they
are going to release a token. This is purely a
phishing attempt… Please guys, let’s continue to be smarter than them guys!
#SCAM
#airdrop
#Bitcoin 
#ETH"""]

category_scores = [doc.cats for doc in nlp.pipe(texts)]
category_scores[0]

In [ ]:
thresh = 0.5
for d in category_scores:
  print(dict((k, v) for k, v in d.items() if v >= thresh))

In [ ]:
# 🚨 SCAM /RIP OFF ALERT 🚨

# ⚠️ Scam warning ⚠️ !
# 🚨Coinbase SCAM ALERT🚨